# MURA XR_WRIST — local study viewer

This optional viewer displays separate radiograph panels from one study **only while running locally or in authorised Colab storage**. It reads internal-train data only and makes no prediction, threshold decision, or model change. Keep `SHOW_RUNTIME_PREVIEW = False` and clear outputs before committing: MURA image pixels must not be stored in Git, including a private repository.

In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Set these local/Colab paths for your authorised MURA copy.
DATA_ROOT = Path('/content/drive/MyDrive/DATA/MURA-v1.1')
MANIFEST_PATH = Path('/content/drive/MyDrive/DATA/WRIST_SPLITS/wrist_manifest_train_only.csv')

# Privacy/governance default: the public notebook never embeds an X-ray output.
SHOW_RUNTIME_PREVIEW = False
STUDY_ID = None  # Optional local study ID. None selects one deterministic internal-train example.
MAX_VIEWS_TO_SHOW = 6
RANDOM_SEED = 42

## Runtime-only preview

When enabled, each real view is shown in its own panel rather than merged into one image. The code does not print patient IDs, study IDs, paths, labels, predictions, or pixel files. It is a visual data-quality check only.

In [ ]:
if SHOW_RUNTIME_PREVIEW:
    if not DATA_ROOT.exists() or not MANIFEST_PATH.exists():
        raise FileNotFoundError('Set DATA_ROOT and MANIFEST_PATH to your authorised local MURA copy.')

    rows = pd.read_csv(MANIFEST_PATH)
    required = {'study_id', 'image_path', 'source_split'}
    missing = required - set(rows.columns)
    if missing:
        raise ValueError(f'Manifest is missing: {sorted(missing)}')
    rows = rows.loc[rows['source_split'].eq('train')].copy()
    rows['image_path'] = rows['image_path'].astype(str).str.replace('\\', '/', regex=False)
    if not rows['image_path'].str.startswith('train/').all():
        raise ValueError('Viewer accepts official-train paths only.')

    grouped = list(rows.sort_values(['study_id', 'image_path']).groupby('study_id', sort=True))
    if STUDY_ID is None:
        candidates = [(key, frame) for key, frame in grouped if 2 <= len(frame) <= MAX_VIEWS_TO_SHOW]
        if not candidates:
            candidates = [(key, frame) for key, frame in grouped if len(frame) >= 1]
        if not candidates:
            raise RuntimeError('No train studies were found in this manifest.')
        selected_key, selected = candidates[RANDOM_SEED % len(candidates)]
    else:
        selected = rows.loc[rows['study_id'].eq(STUDY_ID)].sort_values('image_path')
        if selected.empty:
            raise ValueError('The requested local STUDY_ID is not in the internal-train manifest.')
        selected_key = STUDY_ID

    selected = selected.head(MAX_VIEWS_TO_SHOW).reset_index(drop=True)
    columns = min(3, len(selected))
    rows_count = math.ceil(len(selected) / columns)
    figure, axes = plt.subplots(rows_count, columns, figsize=(4 * columns, 5 * rows_count))
    axes = list(getattr(axes, 'flat', [axes]))
    for view_number, (_, record) in enumerate(selected.iterrows(), start=1):
        with Image.open(DATA_ROOT / record['image_path']) as image:
            axes[view_number - 1].imshow(image.convert('L'), cmap='gray')
        axes[view_number - 1].set_title(f'View {view_number} of {len(selected)}')
        axes[view_number - 1].axis('off')
    for axis in axes[len(selected):]:
        axis.axis('off')
    figure.suptitle('Runtime-only study preview — internal train', y=0.99)
    figure.tight_layout()
    plt.show()

## Before a Git commit

1. Set `SHOW_RUNTIME_PREVIEW = False`.
2. Use **Edit → Clear all outputs** in Colab/Jupyter.
3. Commit the notebook code and the aggregate project charts/tables, never the displayed radiographs.